In [2]:
!python run_experiment2_simple.py \
        --data_dir "/scratch/project_2003370/yueyao/dataset/mini_librispeech/LibriSpeech/dev-clean-2"


=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0.01: expected 1001, got 1001
=== Debug Complete ===


=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0.01: expected 1001, got 1001
=== Debug Complete ===

  ✓ Dataset loader imp

基于最新的日志，**Experiment 2b (基于声纹验证的噪音鲁棒性测试)** 已经圆满成功运行。

此次实验采用了 **“声纹特征验证 (Speaker Verification)”** 的方法，通过计算“同类样本相似度”与“异类样本相似度”之间的 **间距 (Gap)** 来评估模型性能。

以下是详细的实验结果总结与科学分析：

### 1. 核心结论 (Executive Summary)

* **实验成功：** 代码成功解决了之前的维度不匹配问题，通过自动切除分类头并提取特征，获得了有效的评估数据。
* **基准表现 (Clean)：** **MFCC_MLP** 在干净环境下表现显著优于 MEL_MLP。
* MFCC 间距 (Gap): **0.64** (区分度很高)
* Mel 间距 (Gap): **0.38** (区分度一般)


* **噪音表现 (Noisy)：** 两种传统模型在强噪音下均发生 **“灾难性崩溃” (Catastrophic Collapse)**。
* 在 0dB SNR 时，两者的 Gap 都跌至 **~0.02**，意味着模型完全无法区分说话人（特征混在一起）。


* **痛点证明：** 这个结果完美地佐证了你论文的立论——**传统线性特征+简单模型极度缺乏抗噪能力**。

---

### 2. 详细数据分析

**指标说明：** `Gap = (正样本相似度) - (负样本相似度)`。Gap 越大，说明模型把同一个人聚得越紧，把不同人分得越开。

#### A. 干净环境对比 (Clean Baseline)

| 模型 | Positive Sim (同人) | Negative Sim (异人) | **Gap (区分度)** | 评价 |
| --- | --- | --- | --- | --- |
| **MFCC_MLP** | 0.91 | 0.27 | **0.6418** | **优胜**。MFCC 特征更加紧凑、去相关，利于 MLP 学习。 |
| **MEL_MLP** | 0.78 | 0.40 | **0.3834** | 较差。Mel 频谱维度高且包含冗余，简单 MLP 难以提取核心特征。 |

#### B. 噪音环境衰减趋势 (Noise Degradation)

下表展示了 **Gap** 随信噪比 (SNR) 降低而变化的趋势（数值越低越差）：

| 噪音类型 | SNR | MEL_MLP Gap | MFCC_MLP Gap | 现象分析 |
| --- | --- | --- | --- | --- |
| **Gaussian** (白噪音) | 20dB | 0.22 (-42%) | 0.18 (-72%) | **MFCC 对高斯噪音极其敏感**，性能瞬间暴跌。 |
|  | 0dB | 0.02 (失效) | 0.007 (失效) | 两者均完全失效。 |
| **Babble** (人声干扰) | 20dB | 0.25 (-33%) | 0.33 (-48%) | MFCC 在轻微人声干扰下尚可，优于 Mel。 |
|  | 0dB | 0.02 (失效) | 0.01 (失效) | 强人声干扰下，两者均无法提取声纹。 |
| **Cafe/Street** (环境音) | 20dB | ~0.24 | ~0.26 | 表现类似，随噪音增加线性下降。 |
|  | 0dB | ~0.03 | ~0.02 | 彻底淹没。 |

**趋势图解：**
如果画一张折线图 (X轴为 SNR 20->0, Y轴为 Gap)，你会看到：

* **MFCC 线：** 起点很高 (0.64)，但在加入一点噪音 (20dB) 后**陡峭下跌**，斜率极大。
* **Mel 线：** 起点较低 (0.38)，下降相对平缓，但最终殊途同归，都在 0dB 处归零。

---

### 3. 科学解释 (Scientific Interpretation)

这些数据为你的论文提供了强有力的论据：

1. **为什么 MFCC 在干净时更好？**
MFCC 利用离散余弦变换 (DCT) 去除了频谱特征之间的相关性，相当于一种“线性降维”。对于结构简单的 MLP 网络来说，这种特征更容易分类，因此在 Clean 模式下 Gap 达到了 0.64。
2. **为什么两者都不抗噪？**
* **Mel 频谱：** 是原始声学特征，噪音（尤其是 Gaussian/Street）直接叠加在频谱像素上，MLP 只是简单地加权求和，无法区分“这是人声的能量”还是“这是汽车的能量”。
* **MFCC：** 虽然去除了相关性，但也丢弃了许多细微的非线性动力学信息。当噪音混入时，MFCC 系数发生剧烈扰动，导致特征空间中的“簇”被打散。


3. **这对 C-HiLAP (你的混沌模型) 意味着什么？**
Experiment 2 的结果铺平了道路。你已经证明了**“线性特征 (MFCC/Mel) + 线性映射 (MLP) 在噪音面前是脆弱的”**。
接下来，你需要通过 **Experiment 3** 证明：**引入混沌特征 (Lyapunov/维度/熵)** 可以捕捉到语音信号内在的非线性拓扑结构，这种结构在噪音干扰下比频谱能量更稳定，从而在 0dB 时仍能保持较高的 Gap。

### 4. 下一步建议

你现在已经完成了传统基线的完整评估。接下来的步骤非常清晰：

1. **保存数据：** 确保 `verification_results.json` 被妥善保存，这些数字将直接填入你论文的对比表格中。
2. **开始 Experiment 3 (混沌模型)：** 使用完全相同的 `run_experiment2_verification.py` 逻辑（即验证任务逻辑），去测试你的 **C-HiLAP (Chaotic Model)**。
3. **预期对比：** 你希望看到 C-HiLAP 在 Clean 下达到 0.7$~$0.8 的 Gap，更重要的是，在 **0dB 噪音下，Gap 最好能维持在 0.1~0.2 以上**（而不是基线的 0.02），这将是论文的核心亮点。

In [4]:
!python run_experiment2_verification.py


=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0.01: expected 1001, got 1001
=== Debug Complete ===


=== Integration Parameter Debug ===
span=(0.0, 1.0), dt=0.1: expected 11, got 11
span=(0.0, 1.0), dt=0.05: expected 21, got 21
span=(0.0, 1.0), dt=0.01: expected 101, got 101
span=(0.0, 5.0), dt=0.1: expected 51, got 51
span=(0.0, 5.0), dt=0.05: expected 101, got 101
span=(0.0, 5.0), dt=0.01: expected 501, got 501
span=(0.0, 10.0), dt=0.1: expected 101, got 101
span=(0.0, 10.0), dt=0.05: expected 201, got 201
span=(0.0, 10.0), dt=0.01: expected 1001, got 1001
=== Debug Complete ===

  ✓ Dataset loader imp

In [2]:
!python run_experiment2_noise_robustness.py \
    --baseline_checkpoint_dir /scratch/project_2003370/yueyaoao/Model/scl/scripts/outputs/experiment1_unified \
    --chaotic_checkpoint "/scratch/project_2003370/yueyao/Model/experiments/configs/base_chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20260122_131044/checkpoint_epoch_0092.pt" \
    --output_dir ./outputs/experiment2_noise_robustness \
    --all

  ✓ Dataset loader imported successfully
✓ Data package initialized (circular import free)
[2026-01-24 17:37:55] [INFO] Setting global random seed to: 42
[2026-01-24 17:37:55] [INFO] Python random seed set
[2026-01-24 17:37:55] [INFO] PYTHONHASHSEED set to: 42
[2026-01-24 17:37:55] [INFO] NumPy random seed set
[2026-01-24 17:37:55] [INFO] PyTorch CUDA seeds set
[2026-01-24 17:37:55] [INFO] PyTorch strict reproducibility mode enabled
[2026-01-24 17:37:55] [INFO] PyTorch seeds set
[2026-01-24 17:37:55] [WARNING] TensorFlow not available - skipping TensorFlow seed
[2026-01-24 17:37:55] [INFO] Environment information collected
[2026-01-24 17:37:55] [INFO] Configuring deterministic operations
[2026-01-24 17:37:55] [INFO] PyTorch deterministic operations configured
[2026-01-24 17:37:55] [INFO] Starting reproducibility verification
[2026-01-24 17:37:55] [INFO] Setting global random seed to: 42
[2026-01-24 17:37:55] [INFO] Python random seed set
[2026-01-24 17:37:55] [INFO] PYTHONHASHSEED set 

In [ ]:
import json
import os

def create_chaotic_config():
    # 定义配置数据 (字典格式)
    config_data = {
        "num_speakers": 26,
        "sample_rate": 16000,
        "frame_length": 400,
        "hop_length": 160,
        "embedding_dim": 10,
        "delay_method": "autocorr",
        "mlsa_scales": 5,
        "rqa_radius_ratio": 0.1,
        "chaotic_system": "lorenz",
        "evolution_time": 0.5,
        "time_step": 0.01,
        "pooling_type": "comprehensive",
        "speaker_embedding_dim": 256,
        "embedding_hidden_dims": [512, 256, 128],
        "classifier_type": "cosine",
        "optional_modules": {
            "differentiable_features": {"enabled": false},
            "bifurcation_control": {"enabled": false}
        }
    }

    # 指定输出路径
    output_path = "/scratch/project_2003370/yueyao/Model/Relevance_Experiment_of_Thesis/chaotic_26spk_config.json"

    # 写入 JSON 文件
    try:
        # 确保目录存在 (如果 /tmp 不存在，虽然不太可能，但这是一种好习惯)
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        
        with open(output_path, 'w', encoding='utf-8') as f:
            # indent=4 确保输出格式化缩进，而不是一行紧凑的代码
            json.dump(config_data, f, indent=4)
            
        print(f"Success: Configuration file created at {output_path}")
        
    except Exception as e:
        print(f"Error: Failed to create config file. Reason: {e}")

if __name__ == "__main__":
    create_chaotic_config()

# base model best checkpoint

In [9]:
!python run_experiment2_noise_robustness.py \
    --chaotic_checkpoint "/scratch/project_2003370/yueyao/Model/experiments/configs/base_chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20260122_131044/checkpoint_epoch_0092.pt" \
    --all

  ✓ Dataset loader imported successfully
✓ Data package initialized (circular import free)
Core modules imported successfully
[2026-01-24 18:52:43] [INFO] Setting global random seed to: 42
[2026-01-24 18:52:43] [INFO] Python random seed set
[2026-01-24 18:52:43] [INFO] PYTHONHASHSEED set to: 42
[2026-01-24 18:52:43] [INFO] NumPy random seed set
[2026-01-24 18:52:43] [INFO] PyTorch CUDA seeds set
[2026-01-24 18:52:43] [INFO] PyTorch strict reproducibility mode enabled
[2026-01-24 18:52:43] [INFO] PyTorch seeds set
[2026-01-24 18:52:43] [WARNING] TensorFlow not available - skipping TensorFlow seed
[2026-01-24 18:52:43] [INFO] Environment information collected
[2026-01-24 18:52:43] [INFO] Configuring deterministic operations
[2026-01-24 18:52:43] [INFO] PyTorch deterministic operations configured
[2026-01-24 18:52:43] [INFO] Starting reproducibility verification
[2026-01-24 18:52:43] [INFO] Setting global random seed to: 42
[2026-01-24 18:52:43] [INFO] Python random seed set
[2026-01-24 1

In [14]:
!python run_experiment2_noise_robustness.py \
    --baseline_checkpoint_dir /scratch/project_2003370/yueyao/Model/scripts/outputs/experiment1_unified \
    # --chaotic_checkpoint "/scratch/project_2003370/yueyao/Model/experiments/configs/base_chaotic/experiments/chaotic_lorenz_full_chaotic_run_0/checkpoints/exp_20260122_131044/checkpoint_epoch_0092.pt" \
    # --chaotic_config /tmp/config_26.json \
    --baseline_only

  ✓ Dataset loader imported successfully
✓ Data package initialized (circular import free)
Core modules imported successfully
[2026-01-24 18:59:23] [INFO] Setting global random seed to: 42
[2026-01-24 18:59:23] [INFO] Python random seed set
[2026-01-24 18:59:23] [INFO] PYTHONHASHSEED set to: 42
[2026-01-24 18:59:23] [INFO] NumPy random seed set
[2026-01-24 18:59:23] [INFO] PyTorch CUDA seeds set
[2026-01-24 18:59:23] [INFO] PyTorch strict reproducibility mode enabled
[2026-01-24 18:59:23] [INFO] PyTorch seeds set
[2026-01-24 18:59:23] [WARNING] TensorFlow not available - skipping TensorFlow seed
[2026-01-24 18:59:23] [INFO] Environment information collected
[2026-01-24 18:59:23] [INFO] Configuring deterministic operations
[2026-01-24 18:59:23] [INFO] PyTorch deterministic operations configured
[2026-01-24 18:59:23] [INFO] Starting reproducibility verification
[2026-01-24 18:59:23] [INFO] Setting global random seed to: 42
[2026-01-24 18:59:23] [INFO] Python random seed set
[2026-01-24 1